# 75장 · SQLite 백업과 복구 검증

복구를 검증하려면 원본과 분리된 대상에 백업을 복원하고 실제 필드 값을 비교해야 합니다. 행 개수가 같아도 내용은 다를 수 있습니다.

별도 DB에 백업을 복원하고 원본의 소유자·문서·본문을 다시 조회합니다. 서비스 재시작, 백업 파일 생성, 실제 데이터 복구를 각각 다른 결과로 기록합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## backup()으로 별도 데이터베이스에 복원

수업용 DB에는 alice의 과제 안내 문서 하나를 넣습니다. SQLite의 실제 백업 API로 독립된 복사본을 만듭니다. EC2 볼륨 스냅샷이나 PostgreSQL pg_dump를 실행하는 실습과 구분하세요.


### 셀 01 · SQLite 데이터베이스 백업

별개의 DB 객체를 만들고 원본에 커밋한 다음 백업합니다.


In [1]:
import sqlite3
origin = sqlite3.connect(":memory:")
origin.execute("CREATE TABLE documents(owner TEXT, id TEXT, body TEXT)")
with origin:
    origin.execute("INSERT INTO documents VALUES(?,?,?)", ("alice", "doc-1", "과제함에 제출합니다."))
backup = sqlite3.connect(":memory:")
origin.backup(backup)
print(backup.execute("SELECT count(*) FROM documents").fetchone()[0])

1


백업에서 행 하나가 보입니다. 이 단계에서는 별도 복구 대상에서 서비스가 다시 읽을 수 있는지 아직 확인하지 않았습니다.


### 백업본 한 행이 알려주는 것

<p>origin.backup(backup)은 문서 한 행을 가진 원본 DB를 별도 연결에 복사합니다. 대상에서 count(*)가 1이면 행 수를 확인한 셈입니다. 문장의 내용을 확인하려면 그 행을 직접 조회해야 합니다.</p>


### 셀 02 · 빈 DB 복원 후 레코드 대조

원본과 같은 조건으로 조회합니다. 새 대상에는 원래 데이터가 없었습니다.


In [2]:
target = sqlite3.connect(":memory:")
backup.backup(target)
query = "SELECT body FROM documents WHERE owner=? AND id=?"
key = ("alice", "doc-1")
original_row = origin.execute(query, key).fetchone()
restored_row = target.execute(query, key).fetchone()
print(restored_row, restored_row == original_row)

('과제함에 제출합니다.',) True


새 DB에서 읽은 문장과 비교 결과 True가 나옵니다. 조회 조건에 문서 ID뿐 아니라 소유자도 들어가므로 다른 소유자의 같은 이름을 가진 문서가 비교 대상으로 섞이지 않습니다.


### 원본·백업본·복구 대상을 나누기

<p>원본은 origin, 복사본은 backup, 복구 검사를 할 대상은 target입니다. backup.backup(target)을 호출한 뒤 target에만 변경을 가하면 원본을 비교 기준으로 계속 사용할 수 있습니다.</p>


## 행 수와 원문 값 비교

복구 대상의 문장만 다른 상황을 만들어 봅니다. 대상은 버려도 되는 수업용 메모리 DB입니다. 운영 원본을 덮어쓰지 않고도 점검 방식의 차이를 볼 수 있습니다.


### 과제 03 · 원문 필드의 불일치 검출

복원본의 doc-1만 수정한 뒤, 같은 키로 복원본에서 현재 값을 다시 읽으세요. 셀 02에서 읽어 둔 값과 새 조회값이 어떻게 다른지 출력하세요.


In [3]:
# target의 doc-1을 ‘제출 기한 미확인’으로 바꾸세요.
# 셀 02의 restored_row와 새 SELECT 결과를 나란히 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>with target:
    target.execute(&quot;UPDATE documents SET body=? WHERE owner=? AND id=?&quot;,
                   (&quot;제출 기한 미확인&quot;, *key))
current = target.execute(query, key).fetchone()
print(restored_row[0], current[0], sep=&quot; / &quot;)
print(current == original_row)</code></pre>
<p>실행 결과</p><pre><code>과제함에 제출합니다. / 제출 기한 미확인
False</code></pre>
<p>restored_row는 셀 02에서 읽어 둔 튜플입니다. DB를 바꿔도 그 튜플은 바뀌지 않습니다. 변경 뒤에 다시 SELECT한 current와 원본을 대조해야 복원본의 차이가 드러납니다.</p>
</details>


### 같은 키로 원문 비교하기

<p>restored_row와 original_row는 수정 전 값입니다. 수정한 DB에 다시 쿼리를 보내야 현재 본문을 읽을 수 있습니다. 소유자와 문서 ID를 고정하고 본문만 비교하세요.</p>


## 돌아보기

복원본에서 잘못된 문장이 돌아왔는데도 검사가 통과했다면, 비교에 쓴 두 값의 조회 시점을 보세요. 변경 전에 읽은 두 튜플을 다시 비교하면 변경 후 DB의 오류를 놓칩니다.
